# NIH14 Concept Intervention

Zero or boost specific concepts and measure the downstream effect on each pathology's binary classifier. This mirrors the intervention plots from the Label-Free CBM paper but loops over NIH14 classes.


In [ ]:
import os
import json
from pathlib import Path

import torch
import numpy as np

os.chdir('..')

import cbm
import data_utils
from evaluate_cbm_nih import _load_thresholds


In [ ]:
MODEL_DIR = 'saved_models/nih14_example'
DEVICE = 'cuda'
CLASS_NAME = 'Pneumonia'
THRESHOLDS_PATH = 'saved_models/nih14_example/thresholds.json'
THRESHOLD_FALLBACK = 0.5


In [ ]:
model = cbm.load_cbm(MODEL_DIR, DEVICE)
model.eval()

with open(Path(MODEL_DIR) / 'args.txt') as f:
    args = json.load(f)
classes_path = data_utils.LABEL_FILES['nih14']
with open(classes_path) as f:
    classes = [c for c in f.read().splitlines() if c]
class_idx = classes.index(CLASS_NAME)
print(f'Loaded model covering {len(classes)} pathologies. Editing class {CLASS_NAME} (index {class_idx}).')

try:
    thresholds = _load_thresholds(THRESHOLDS_PATH, classes)
    threshold_vec = thresholds
except Exception as exc:
    threshold_vec = None
    print('Threshold file missing; falling back to scalar threshold when evaluating.', exc)


In [ ]:
from torch.utils.data import DataLoader
from tqdm import tqdm

split_name = 'nih14_val'
views = ['PA']

data_utils.configure_nih_dataset(img_dir=args['nih_img_dir'],
                                 csv_path=args.get('nih_csv_path'),
                                 train_fraction=args.get('nih_train_fraction', 0.9),
                                 split_seed=args.get('nih_split_seed', 0),
                                 views=views)
_, preprocess = data_utils.get_target_model(args['backbone'], 'cpu')
dataset = data_utils.get_data(split_name, preprocess)
loader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

logits_list, concept_list, targets = [], [], []
with torch.no_grad():
    for images, label in tqdm(loader):
        images = images.to(DEVICE)
        logits, concept_act = model(images)
        logits_list.append(logits.cpu())
        concept_list.append(concept_act.cpu())
        targets.append(label[:, class_idx])

logits = torch.cat(logits_list)
concept_act = torch.cat(concept_list)
targets = torch.cat(targets).numpy()
print('Cached logits', logits.shape, 'concept activations', concept_act.shape)


## Intervention helpers

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

def forward_with_concepts(concepts):
    with torch.no_grad():
        logits = model.head(model.project(concepts.to(DEVICE)))
    return torch.sigmoid(logits[:, class_idx]).cpu().numpy()

baseline_probs = forward_with_concepts(concept_act.clone())
baseline = {
    'auroc': roc_auc_score(targets, baseline_probs),
    'ap': average_precision_score(targets, baseline_probs)
}
print('Baseline:', baseline)

def run_intervention(modifier):
    edited = concept_act.clone()
    edited = modifier(edited)
    probs = forward_with_concepts(edited)
    return {
        'auroc': roc_auc_score(targets, probs),
        'ap': average_precision_score(targets, probs)
    }

def zero_concepts(idx_list):
    def hook(t):
        t[:, idx_list] = 0
        return t
    return hook

def boost_concepts(idx_list, factor=2.0):
    def hook(t):
        t[:, idx_list] *= factor
        return t
    return hook

print('Zero concept 0:', run_intervention(zero_concepts([0])))
print('Boost concept 0:', run_intervention(boost_concepts([0], 1.5)))
